# Module 5.5: Evaluating a Language Model

You can now **train** a model (5.3) and **sample** text from it (5.4) — but how do you know if it's actually any *good*? This notebook teaches you to **measure** a language model: perplexity, train-vs-validation loss, qualitative checks, and how the big labs benchmark real LLMs.

## 1. The question: we minimized loss — but is the model GOOD?

In Module 5.2 we ran the sacred training loop and watched the loss number drop. Job done, right?

Not quite. A dropping loss tells you the model is *getting better at the exact text you showed it*. But that's like a student whose grade goes up — is it because they actually learned the subject, or because they memorized last year's answer key? We need an external, repeatable way to **measure quality** that we can compare across models.

This notebook is your model's **report card**. We'll cover:
1. **Perplexity** — turning loss into an intuitive "how confused is the model?" score.
2. **Comparison** — perplexity is only meaningful next to a baseline (a random model).
3. **Train vs. validation loss** — catching the model when it *memorizes* instead of *learns*.
4. **Eyeballing samples** — because numbers don't capture everything.
5. **Benchmarks** — how real LLMs are scored on knowledge and reasoning.
6. **The limits** — what perplexity flatly cannot tell you.

First, let's train a tiny real model so we have something to grade.

### Setup: train a tiny char-level model

To evaluate a model, we need a model. We'll quick-train a small **character-level** GPT on a short embedded poem (no downloads needed) for a few hundred iterations. This takes well under two minutes.

Crucially, we split the poem into a **training** region and a separate **held-out validation** region. The model never trains on the validation text — that's what makes it a fair test (this is the held-out-data idea from Module 0.2).

In [ ]:
import math
import torch
import matplotlib.pyplot as plt
from llm_workout.model import GPT

torch.manual_seed(0)

# --- A small embedded text corpus (Hamlet's soliloquy). No network needed. ---
corpus = """To be, or not to be, that is the question:
Whether tis nobler in the mind to suffer
The slings and arrows of outrageous fortune,
Or to take arms against a sea of troubles
And by opposing end them. To die-to sleep,
No more; and by a sleep to say we end
The heart-ache and the thousand natural shocks
That flesh is heir to: tis a consummation
Devoutly to be wishd. To die, to sleep;
To sleep, perchance to dream-ay, theres the rub:
For in that sleep of death what dreams may come,
When we have shuffled off this mortal coil,
Must give us pause. Theres the respect
That makes calamity of so long life.
"""

# --- Build a character-level vocabulary ---
chars = sorted(set(corpus))
vocab_size = len(chars)
stoi = {c: i for i, c in enumerate(chars)}
itos = {i: c for c, i in stoi.items()}
encode = lambda s: torch.tensor([stoi[c] for c in s], dtype=torch.long)
decode = lambda t: "".join(itos[int(i)] for i in t)

# --- Hold out a validation split of DISTINCT text the model never trains on. ---
# First 80% of the (unique) poem is training; the last 20% is held out for eval.
split = int(0.8 * len(corpus))
train_text, val_text = corpus[:split], corpus[split:]
# Repeat each so there are enough characters to draw batches from.
train_data = encode(train_text * 8)
val_data = encode(val_text * 4)

print(f"Vocabulary size: {vocab_size} unique characters")
print(f"Train chars: {len(train_data)} | Val chars: {len(val_data)} (held out, distinct text)")

In [ ]:
# --- A small model and a batch sampler ---
block_size = 48  # how many characters of context the model sees per training window

# max_seq_len is the longest position the model can handle. It must cover BOTH a
# training window AND the longer prompt+generation run we do in Section 5, so we
# give it generous headroom.
max_seq_len = 256

def get_batch(data, batch_size=16):
    """Grab random (context -> next-char) windows, shifted by 1 (Module 5.1)."""
    ix = torch.randint(0, len(data) - block_size - 1, (batch_size,))
    x = torch.stack([data[i:i + block_size] for i in ix])
    y = torch.stack([data[i + 1:i + block_size + 1] for i in ix])
    return x, y

model = GPT(
    vocab_size=vocab_size,
    d_model=96,
    num_layers=3,
    num_heads=4,
    hidden_dim=256,
    max_seq_len=max_seq_len,
)
n_params = sum(p.numel() for p in model.parameters())
print(f"Model created with {n_params:,} parameters (a *tiny* GPT).")

In [ ]:
@torch.no_grad()
def estimate_loss(data, iters=30):
    """Average cross-entropy loss over several random batches (smooths noise)."""
    was_training = model.training
    model.eval()
    losses = []
    for _ in range(iters):
        x, y = get_batch(data)
        _, loss, _ = model(x, y)  # GPT.forward returns (logits, loss, kv_caches)
        losses.append(loss.item())
    model.train(was_training)
    return sum(losses) / len(losses)

# Loss of the FRESH, untrained model (we'll use this as a baseline in Section 3).
random_train_loss = estimate_loss(train_data)
random_val_loss = estimate_loss(val_data)
print(f"Untrained model -- train loss: {random_train_loss:.3f} | val loss: {random_val_loss:.3f}")

# --- Train, recording train & val loss at checkpoints so we can plot later. ---
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-3)
checkpoints, train_curve, val_curve = [], [], []

max_iters = 800
for it in range(max_iters):
    if it % 80 == 0:
        checkpoints.append(it)
        train_curve.append(estimate_loss(train_data))
        val_curve.append(estimate_loss(val_data))
    x, y = get_batch(train_data)
    _, loss, _ = model(x, y)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

# Final readings
checkpoints.append(max_iters)
train_curve.append(estimate_loss(train_data))
val_curve.append(estimate_loss(val_data))
print(f"Trained model   -- train loss: {train_curve[-1]:.3f} | val loss: {val_curve[-1]:.3f}")
print("Loss dropped a lot. But how good is that, really? Let's make it meaningful.")

## 2. From cross-entropy loss to perplexity

Recall from Module 5.1: cross-entropy loss is the model's average **surprise** at the true next token, measured in *nats* (natural-log units). A loss of `0.7` is lower than `2.3`, but neither number means much on its own.

**Perplexity** makes the number speak human:

$$\text{perplexity} = e^{\text{cross-entropy loss}}$$

The intuition: perplexity is **the effective number of equally-likely choices the model is torn between for each next token.**

- Perplexity of **1** → the model is certain every time (it knows exactly what comes next). Perfect.
- Perplexity of **5** → on average the model is as confused as if it had to guess among 5 equally-likely options.
- Perplexity of **38** → it's basically guessing uniformly among all ~38 characters in our vocab. Useless.

**Lower is better.** Let's compute our trained model's perplexity. We'll start by measuring it on the **training text** — text the model has actually been learning from — to confirm training did its job. (In Section 4 we'll switch to *held-out* text for the real, unforgiving test.)

In [ ]:
def perplexity(loss):
    """Perplexity is just exp() of the cross-entropy loss."""
    return math.exp(loss)

trained_train_loss = estimate_loss(train_data)
print(f"Trained model on its TRAINING text:")
print(f"  cross-entropy loss = {trained_train_loss:.3f} nats")
print(f"  perplexity         = {perplexity(trained_train_loss):.2f}")
print()
print(f"Read it as: 'for each character, the model is torn between about")
print(f"{perplexity(trained_train_loss):.2f} equally-likely options on average.'")
print("Close to 1.0 means the model is almost never surprised by this text.")

## 3. Make the number meaningful: compare against a baseline

Is a perplexity of (say) 7 good or bad? You can't know without a yardstick.

The simplest yardstick is a **random, untrained model**. A freshly initialized network has no idea what comes next, so it spreads probability roughly evenly over all `vocab_size` tokens. Its perplexity should land near `vocab_size` itself — the "I'm just guessing uniformly" score.

Let's contrast the untrained baseline with our trained model **on the same training text** and see how dramatically training collapses the perplexity. (We measured the untrained model's loss back in Section 1, before any weight updates.)

In [ ]:
random_ppl = perplexity(random_train_loss)    # untrained, measured in Section 1
trained_ppl = perplexity(trained_train_loss)  # trained, measured in Section 2

print(f"Vocabulary size (uniform-guess yardstick): {vocab_size}")
print(f"RANDOM  (untrained) model perplexity: {random_ppl:7.2f}   <- ~ vocab_size, pure guessing")
print(f"TRAINED model perplexity            : {trained_ppl:7.2f}   <- much lower!")
print(f"\nTraining shrank perplexity by {random_ppl / trained_ppl:.0f}x.")
print("That collapse -- from 'guessing among all characters' down toward 'certain' --")
print("is the whole point of training. The baseline is what makes the win visible.")

We can also watch perplexity *fall over training* using the checkpoints we recorded. This is the curve every ML practitioner stares at while a model trains.

In [ ]:
train_ppl_curve = [perplexity(l) for l in train_curve]
val_ppl_curve = [perplexity(l) for l in val_curve]

plt.figure(figsize=(8, 4.5))
plt.plot(checkpoints, train_ppl_curve, marker="o", label="train perplexity")
plt.axhline(vocab_size, color="gray", linestyle="--", label=f"random baseline (~vocab={vocab_size})")
plt.xlabel("training iteration")
plt.ylabel("perplexity (lower = better)")
plt.title("Perplexity collapses as the model learns the text")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## 4. Train vs. validation loss: catching overfitting

That near-1.0 training perplexity looked like a triumph. But here's the catch: a low *training* perplexity might mean the model **learned the language**... or it might mean the model **memorized the training text** word-for-word without understanding anything. These look identical if you only watch training loss!

The tell-tale sign is the gap between train and **validation** (held-out) performance — the held-out region we carved off at the start, which the model has *never* trained on:

- **Low train loss AND low val loss** → the model genuinely *generalized*. 🎉
- **Low train loss BUT high val loss** → the model *memorized* the training set and falls apart on text it hasn't seen. This is **overfitting**. (This is exactly why Module 0.2 insisted on holding out data.)

Our tiny model trains on a small amount of text, so it's prone to memorizing. Let's plot both curves and look for the gap.

In [ ]:
plt.figure(figsize=(8, 4.5))
plt.plot(checkpoints, train_curve, marker="o", label="train loss")
plt.plot(checkpoints, val_curve, marker="s", label="validation loss (held out)")
plt.xlabel("training iteration")
plt.ylabel("cross-entropy loss")
plt.title("Train vs. validation loss: the gap reveals memorization")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

print(f"Final train loss: {train_curve[-1]:.3f}  (perplexity {perplexity(train_curve[-1]):.2f})")
print(f"Final val   loss: {val_curve[-1]:.3f}  (perplexity {perplexity(val_curve[-1]):.2f})")
gap = val_curve[-1] - train_curve[-1]
print(f"\nThe val loss is much higher than the train loss (gap = {gap:.2f}).")
print("Low train + high val = the model MEMORIZED its training text instead of")
print("learning to generalize. THIS is why we always hold out a validation set:")
print("without it, the near-perfect train perplexity would have fooled us completely.")

> **Note:** Our model overfits *on purpose* here — it's tiny and the dataset is tiny, so memorization is easy to provoke and instructive to see. Real training fights overfitting with vastly more data, regularization (like the weight decay in AdamW), dropout, and early stopping (halt when val loss starts climbing).

## 5. Qualitative evaluation: just read the output

Metrics are necessary but not sufficient. Two models with the same perplexity can produce wildly different *feeling* text. The oldest evaluation trick in the book is still essential: **sample some output and read it with your own eyes.**

Let's generate a few characters from our trained model (using `model.generate` from Module 5.4) and judge it ourselves.

In [ ]:
torch.manual_seed(0)
prompt = "To be"
context = encode(prompt).unsqueeze(0)  # shape (1, T)

generated = list(model.generate(context, max_new_tokens=120))
sample = prompt + decode(torch.tensor(generated))
print("--- Model's sample ---")
print(sample)
print("----------------------")
print("\nEyeball check: does it look like English-ish Shakespeare? Are words")
print("spelled plausibly? Since the model memorized the training text, you'll")
print("likely see it parroting familiar phrases. A low perplexity number alone")
print("would NOT have told you whether the text reads coherently -- only reading it does.")

Perplexity says "the model predicts the next character well." It says **nothing** about whether the output is coherent over long spans, factually correct, or on-topic. For that, humans (and these days, other LLMs acting as judges) read the samples. Quantitative + qualitative together give the full picture.

## 6. How real LLMs are benchmarked (conceptual)

Perplexity grades "can you predict text?" But when you read that GPT-4 or Llama 3 "scored 86% on MMLU," that's a different kind of test: a **benchmark** — a fixed suite of tasks with known right answers, so different models can be compared apples-to-apples.

A few famous ones:

- **MMLU** (Massive Multitask Language Understanding) — thousands of multiple-choice questions across 57 subjects (history, law, medicine, math...). Tests **knowledge**.
- **HellaSwag** — pick the most sensible ending to a short scenario. Tests **commonsense reasoning**.
- **GSM8K** — grade-school math word problems. Tests **multi-step reasoning**.

**The clever bit — how a multiple-choice question is scored without the model "clicking" an answer:** you show the model the question followed by each candidate answer, and measure the probability (or, equivalently, the loss/perplexity) the model assigns to each completion. The answer the model finds **least surprising** — highest probability — is its "choice." Score = fraction of questions where that choice is correct.

Notice this is the *exact same machinery* you just used: cross-entropy / probability of a sequence. Benchmarks are perplexity-style measurement pointed at carefully-designed questions instead of random held-out text. (No code here — just the intuition.)

## 7. The limits of perplexity

Perplexity is the workhorse of language-model evaluation, but be clear-eyed about what it does **not** measure. Perplexity tells you how well the model **predicts held-out text**. It does **not** tell you whether the model is:

- **Helpful** — a model can have great perplexity and still ignore what you actually asked.
- **Truthful** — fluent text can be confidently, eloquently *wrong* (hallucination). Predicting plausible-sounding tokens is not the same as being factual.
- **Safe / harmless** — perplexity has no notion of whether the output is toxic, biased, or dangerous.
- **Well-grounded** — whether claims are actually supported by a given source document.

Measuring those qualities — **generation quality, hallucination, groundedness, and safety** — is a deeper subject. It needs human preference data, reference-based metrics, retrieval checks, and LLM-as-judge pipelines, and it belongs to an **applied-LLM evaluation track** beyond this from-scratch course. (We won't implement it here — just know it exists and that perplexity is only the first rung of the ladder.)

## Summary

You added the final missing skill to your toolkit:

- **Perplexity = exp(cross-entropy loss)** — the effective number of choices the model is torn between. Lower is better.
- A perplexity number is only meaningful **against a baseline**; a random model sits near `vocab_size`, and training collapses it.
- The **train-vs-validation gap** exposes memorization (overfitting) — the whole reason we hold out data.
- **Read the samples** — metrics miss coherence and factuality.
- Real LLMs are scored on **benchmarks** (MMLU, HellaSwag, ...) by checking whether the correct answer gets the highest probability.
- Perplexity does **not** measure helpfulness, truthfulness, or safety — that's a deeper, applied topic.

**You can now train, sample from, AND measure a model.** 🎓

### 🏋️ Try it yourself

Get hands-on with the report card:

1. **Bigger or smaller model.** Re-create `model` with `num_layers=1` (or bump `d_model`). Retrain and compare the final **validation** perplexity. Does more capacity help generalization here, or just deepen the overfitting gap?
2. **Watch overfitting grow.** Crank `max_iters` up (e.g. 2000) and re-plot train vs. val loss. Does validation loss eventually *rise* even as train loss keeps falling? That upturn is the textbook overfitting signal — and the moment **early stopping** would halt training.
3. **Verify the random baseline.** Make a brand-new untrained `GPT`, measure its perplexity on `val_data`, and confirm it lands near `vocab_size`. Why `vocab_size` and not something else? (Hint: $e^{\ln(\text{vocab\_size})} = \text{vocab\_size}$ — uniform guessing.)

In [ ]:
# Your turn! Starter for experiment 3: confirm a random model's perplexity ~ vocab_size.
torch.manual_seed(123)
fresh_model = GPT(
    vocab_size=vocab_size, d_model=96, num_layers=3,
    num_heads=4, hidden_dim=256, max_seq_len=max_seq_len,
)

@torch.no_grad()
def quick_ppl(m, data, iters=30):
    m.eval()
    losses = []
    for _ in range(iters):
        x, y = get_batch(data)
        _, loss, _ = m(x, y)
        losses.append(loss.item())
    return math.exp(sum(losses) / len(losses))

print(f"Fresh model perplexity on val_data: {quick_ppl(fresh_model, val_data):.2f}")
print(f"vocab_size (expected ballpark)    : {vocab_size}")

# Now try experiments 1 and 2 above. Happy measuring!